# LSTM and GRU Model Training for Aircraft Engine RUL Prediction

This notebook trains deep learning models to predict Remaining Useful Life (RUL)
for aircraft engines using multivariate time-series sensor data.

The models use sequences generated from the NASA C-MAPSS turbofan dataset.

### Objectives

• Load prepared time-series sequences  
• Train LSTM and GRU deep learning models  
• Evaluate model performance  
• Compare model accuracy  
• Save trained models for deploymenttrained models

In [15]:
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

## Load Prepared Sequence Data

The previous notebook generated time-series sequences for training.

Each sample has the structure:

X shape:
(number_of_samples, time_steps, number_of_features)

Target variable:
Remaining Useful Life (RUL)

In [16]:
BASE_DIR = Path(r"C:\Users\Kal\aircraft-engine-safety-risk-prediction")

DATA_DIR = BASE_DIR / "data" / "processed"

X_fd004_train = np.load(DATA_DIR / "X_fd004_train.npy")
y_fd004_train = np.load(DATA_DIR / "y_fd004_train.npy")

X_fd004_val = np.load(DATA_DIR / "X_fd004_val.npy")
y_fd004_val = np.load(DATA_DIR / "y_fd004_val.npy")

X_test = np.load(DATA_DIR / "X_fd004_test.npy")
y_test = np.load(DATA_DIR / "y_fd004_test.npy")

print(X_fd004_train.shape, y_fd004_train.shape)
print(X_fd004_val.shape, y_fd004_val.shape)

(43523, 30, 63) (43523,)
(10505, 30, 63) (10505,)


## Build LSTM Model

Long Short-Term Memory (LSTM) networks are widely used for
time-series prediction problems.

LSTM can capture long-term temporal dependencies in sensor signals,
making it suitable for degradation modeling in predictive maintenance.

In [17]:
def build_lstm(input_shape):

    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),

        LSTM(32),
        Dropout(0.2),

        Dense(16, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    return model

## Train LSTM Model

We train the model using early stopping to prevent overfitting.
Training stops automatically if validation performance stops improving.

In [18]:
lstm_model = build_lstm(X_train.shape[1:])

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_lstm = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=128,
    callbacks=[early_stop]
)

NameError: name 'X_train' is not defined

## Build GRU Model

Gated Recurrent Units (GRU) are a simplified version of LSTM.

They require fewer parameters and often train faster while still
capturing temporal dependencies in time-series data.

In [ ]:
def build_gru(input_shape):

    model = Sequential([
        GRU(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),

        GRU(32),
        Dropout(0.2),

        Dense(16, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    return model

## Train GRU Model

The GRU model is trained using the same training configuration
to allow a fair comparison with the LSTM model.

In [ ]:
gru_model = build_gru(X_train.shape[1:])

history_gru = gru_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=128,
    callbacks=[early_stop]
)

## Evaluate Model Performance

We evaluate models using:

MAE (Mean Absolute Error)

RMSE (Root Mean Squared Error)

These metrics measure prediction accuracy for Remaining Useful Life.

In [ ]:
y_pred_lstm = lstm_model.predict(X_test).flatten()
y_pred_gru = gru_model.predict(X_test).flatten()

mae_lstm = mean_absolute_error(y_test, y_pred_lstm)
rmse_lstm = np.sqrt(mean_squared_error(y_test, y_pred_lstm))

mae_gru = mean_absolute_error(y_test, y_pred_gru)
rmse_gru = np.sqrt(mean_squared_error(y_test, y_pred_gru))

print("LSTM MAE:", mae_lstm)
print("LSTM RMSE:", rmse_lstm)

print("GRU MAE:", mae_gru)
print("GRU RMSE:", rmse_gru)

## Visualization of Predictions

We compare predicted RUL values against true values to visually
assess prediction performance.

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(y_test[:200], label="True RUL")
plt.plot(y_pred_lstm[:200], label="LSTM Prediction")

plt.legend()
plt.title("RUL Prediction vs True Values")

plt.show()

## Save Trained Models

Saving trained models allows reuse for:

• deployment
• inference
• risk scoring

In [ ]:
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

lstm_model.save(MODEL_DIR / "lstm_rul_model.h5")
gru_model.save(MODEL_DIR / "gru_rul_model.h5")

print("Models saved successfully")

In [ ]:
# Summary

This notebook trained deep learning models for aircraft engine
Remaining Useful Life prediction.

Completed tasks:

✔ LSTM model training  
✔ GRU model training  
✔ Performance evaluation  
✔ Model comparison  
✔ Visualization of predictions  
✔ Model saving for future use

Next step:

Risk scoring and fleet-level safety analysis.